# Principal Component Analysis: Visual Guide

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/methods-algorithms/blob/master/notebooks/L05_pca.ipynb)

**Learning Objectives:**
- Understand how PCA finds directions of maximum variance via eigendecomposition
- Visualize scree plots and reconstruction error to select the number of components
- See when PCA fails on non-linear data (Swiss roll)
- Use PCA as preprocessing to improve classification accuracy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.datasets import load_iris, make_classification, make_swiss_roll
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

np.random.seed(42)

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

# ML color palette
ML_PURPLE = '#3333B2'
ML_BLUE = '#0066CC'
ML_ORANGE = '#FF7F0E'
ML_GREEN = '#2CA02C'
ML_RED = '#D62728'

CLASS_COLORS = [ML_BLUE, ML_ORANGE, ML_GREEN]

print('Setup complete.')

## 1. Load and Explore Data

In [ ]:
# Load Iris dataset (4 features, 3 classes, 150 samples)
iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f'Shape: {X.shape}')
print(f'Features: {feature_names}')
print(f'Classes: {list(target_names)}')
print(f'Class distribution: {np.bincount(y)}')

# Visual 1: Scatter matrix of 4 features
fig, axes = plt.subplots(4, 4, figsize=(12, 10))
for i in range(4):
    for j in range(4):
        ax = axes[i, j]
        if i == j:
            for k in range(3):
                ax.hist(X[y == k, i], bins=15, alpha=0.5,
                        color=CLASS_COLORS[k], label=target_names[k])
        else:
            for k in range(3):
                ax.scatter(X[y == k, j], X[y == k, i], s=15, alpha=0.6,
                           color=CLASS_COLORS[k])
        if i == 3:
            ax.set_xlabel(feature_names[j][:10], fontsize=8)
        if j == 0:
            ax.set_ylabel(feature_names[i][:10], fontsize=8)
        ax.tick_params(labelsize=6)

handles = [mpatches.Patch(color=CLASS_COLORS[k], label=target_names[k]) for k in range(3)]
fig.legend(handles=handles, loc='upper right', fontsize=10)
fig.suptitle('Iris Dataset: Pairwise Feature Scatter Matrix', fontsize=14)
plt.tight_layout(rect=[0, 0, 0.95, 0.96])
plt.show()

## 2. PCA: Finding Principal Axes

In [ ]:
# Standardize and fit PCA with all 4 components
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca_full = PCA(n_components=4)
X_pca = pca_full.fit_transform(X_scaled)

evr = pca_full.explained_variance_ratio_
cumvar = np.cumsum(evr)

# Visual 2: Scree plot (bar + cumulative line)
fig, ax = plt.subplots(figsize=(10, 6))
x_pos = np.arange(1, 5)
bars = ax.bar(x_pos, evr, color=ML_PURPLE, alpha=0.7, label='Individual')
ax.plot(x_pos, cumvar, 'o-', color=ML_ORANGE, linewidth=2, markersize=8,
        label='Cumulative')
ax.axhline(y=0.90, color=ML_RED, linestyle='--', linewidth=1.5, label='90% threshold')

for i, (v, c) in enumerate(zip(evr, cumvar)):
    ax.text(x_pos[i], v + 0.01, f'{v:.1%}', ha='center', fontsize=10, color=ML_PURPLE)
    ax.text(x_pos[i] + 0.15, c + 0.02, f'{c:.1%}', fontsize=9, color=ML_ORANGE)

ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance Ratio')
ax.set_title('Scree Plot: Explained Variance per Component')
ax.set_xticks(x_pos)
ax.set_xticklabels(['PC1', 'PC2', 'PC3', 'PC4'])
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Explained variance ratios: {evr.round(4)}')
print(f'Cumulative: {cumvar.round(4)}')

In [ ]:
# Visual 3: Biplot (PC1 vs PC2 with loading arrows)
fig, ax = plt.subplots(figsize=(10, 6))

for k in range(3):
    mask = y == k
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], s=50, alpha=0.6,
               color=CLASS_COLORS[k], label=target_names[k], edgecolors='white')

# Loading arrows
loadings = pca_full.components_[:2, :].T  # (4 features, 2 PCs)
scale = 3.0  # scale arrows for visibility
for i, fname in enumerate(feature_names):
    ax.annotate('', xy=(loadings[i, 0] * scale, loadings[i, 1] * scale),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=ML_RED, linewidth=2))
    ax.text(loadings[i, 0] * scale * 1.15, loadings[i, 1] * scale * 1.15,
            fname, fontsize=9, color=ML_RED, ha='center')

ax.set_xlabel(f'PC1 ({evr[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({evr[1]:.1%} variance)')
ax.set_title('PCA Biplot: Iris Data with Loading Arrows')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
plt.tight_layout()
plt.show()

## 3. How Many Components?

In [ ]:
# Visual 4: Cumulative variance with threshold lines
fig, ax = plt.subplots(figsize=(10, 6))
x_pos = np.arange(1, 5)

ax.plot(x_pos, cumvar, 'o-', color=ML_PURPLE, linewidth=2.5, markersize=10)
ax.axhline(y=0.90, color=ML_ORANGE, linestyle='--', linewidth=2, label='90% threshold')
ax.axhline(y=0.95, color=ML_RED, linestyle='--', linewidth=2, label='95% threshold')

# Mark where thresholds are crossed
k_90 = np.searchsorted(cumvar, 0.90) + 1
k_95 = np.searchsorted(cumvar, 0.95) + 1
ax.scatter([k_90], [cumvar[k_90 - 1]], s=200, marker='*', color=ML_ORANGE, zorder=5)
ax.annotate(f'{k_90} PCs for 90%', xy=(k_90, cumvar[k_90 - 1]),
            xytext=(k_90 + 0.3, cumvar[k_90 - 1] - 0.05),
            fontsize=11, arrowprops=dict(arrowstyle='->', color='black'))
ax.scatter([k_95], [cumvar[k_95 - 1]], s=200, marker='*', color=ML_RED, zorder=5)
ax.annotate(f'{k_95} PCs for 95%', xy=(k_95, cumvar[k_95 - 1]),
            xytext=(k_95 + 0.3, cumvar[k_95 - 1] - 0.05),
            fontsize=11, arrowprops=dict(arrowstyle='->', color='black'))

for i, c in enumerate(cumvar):
    ax.text(x_pos[i], c + 0.015, f'{c:.1%}', ha='center', fontsize=10)

ax.set_xlabel('Number of Components')
ax.set_ylabel('Cumulative Explained Variance')
ax.set_title('How Many Components to Keep?')
ax.set_xticks(x_pos)
ax.set_ylim(0.5, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Components for 90% variance: {k_90}')
print(f'Components for 95% variance: {k_95}')

In [ ]:
# Visual 5: Reconstruction error vs n_components
n_components_range = range(1, 5)
recon_errors = []

for k in n_components_range:
    pca_k = PCA(n_components=k)
    X_proj = pca_k.fit_transform(X_scaled)
    X_recon = pca_k.inverse_transform(X_proj)
    error = np.mean((X_scaled - X_recon) ** 2)
    recon_errors.append(error)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(list(n_components_range), recon_errors, 'o-', color=ML_PURPLE,
        linewidth=2.5, markersize=10)
ax.fill_between(list(n_components_range), recon_errors, alpha=0.15, color=ML_PURPLE)

for i, (k, e) in enumerate(zip(n_components_range, recon_errors)):
    ax.text(k, e + 0.02, f'{e:.3f}', ha='center', fontsize=10)

ax.set_xlabel('Number of Components')
ax.set_ylabel('Mean Squared Reconstruction Error')
ax.set_title('Reconstruction Error vs. Number of Components')
ax.set_xticks(list(n_components_range))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. PCA on High-Dimensional Data

In [ ]:
# Visual 6: PCA on 20-feature synthetic data
X_hd, y_hd = make_classification(
    n_samples=300, n_features=20, n_informative=5,
    n_redundant=5, n_classes=3, n_clusters_per_class=1,
    random_state=42
)
X_hd_scaled = StandardScaler().fit_transform(X_hd)
pca_hd = PCA(n_components=2)
X_hd_pca = pca_hd.fit_transform(X_hd_scaled)

fig, ax = plt.subplots(figsize=(10, 6))
for k in range(3):
    mask = y_hd == k
    ax.scatter(X_hd_pca[mask, 0], X_hd_pca[mask, 1], s=50, alpha=0.6,
               color=CLASS_COLORS[k], label=f'Class {k}', edgecolors='white')

ax.set_xlabel(f'PC1 ({pca_hd.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca_hd.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('PCA on 20-Feature Synthetic Data (Reduced to 2D)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Original shape: {X_hd.shape}')
print(f'Top 2 PCs explain: {pca_hd.explained_variance_ratio_.sum():.1%} of variance')

## 5. PCA Limitation: Non-Linear Data

In [ ]:
# Visual 7: PCA on Swiss roll (failure case)
X_swiss, t_swiss = make_swiss_roll(n_samples=1000, noise=0.5, random_state=42)
X_swiss_scaled = StandardScaler().fit_transform(X_swiss)

pca_swiss = PCA(n_components=2)
X_swiss_pca = pca_swiss.fit_transform(X_swiss_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 3D view (as 2D projection of first two original dims)
ax = axes[0]
sc = ax.scatter(X_swiss[:, 0], X_swiss[:, 2], c=t_swiss, cmap='Spectral',
                s=15, alpha=0.7)
ax.set_title('Swiss Roll (Original: Dim 1 vs Dim 3)')
ax.set_xlabel('Dimension 1')
ax.set_ylabel('Dimension 3')
plt.colorbar(sc, ax=ax, label='Position on roll')

# PCA projection
ax = axes[1]
sc = ax.scatter(X_swiss_pca[:, 0], X_swiss_pca[:, 1], c=t_swiss, cmap='Spectral',
                s=15, alpha=0.7)
ax.set_title('PCA Projection (2D) -- Colors Interleave!')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
plt.colorbar(sc, ax=ax, label='Position on roll')

plt.suptitle('PCA Fails on Non-Linear Data: Swiss Roll', fontsize=14)
plt.tight_layout()
plt.show()

## 6. PCA as Preprocessing

In [ ]:
# Visual 8: Compare raw vs PCA preprocessing for classification
X_clf, y_clf = make_classification(
    n_samples=500, n_features=50, n_informative=10,
    n_redundant=20, n_classes=2, random_state=42
)

# Raw features (scaled)
X_clf_scaled = StandardScaler().fit_transform(X_clf)
scores_raw = cross_val_score(LogisticRegression(max_iter=1000, random_state=42),
                             X_clf_scaled, y_clf, cv=5, scoring='accuracy')

# PCA preprocessing (keep 95% variance)
pca_pre = PCA(n_components=0.95)
X_clf_pca = pca_pre.fit_transform(X_clf_scaled)
scores_pca = cross_val_score(LogisticRegression(max_iter=1000, random_state=42),
                             X_clf_pca, y_clf, cv=5, scoring='accuracy')

# PCA with 2 components only
pca_2 = PCA(n_components=2)
X_clf_pca2 = pca_2.fit_transform(X_clf_scaled)
scores_pca2 = cross_val_score(LogisticRegression(max_iter=1000, random_state=42),
                              X_clf_pca2, y_clf, cv=5, scoring='accuracy')

fig, ax = plt.subplots(figsize=(10, 6))
labels = [f'Raw\n(50 features)', f'PCA 95%\n({pca_pre.n_components_} PCs)', 'PCA\n(2 PCs)']
means = [scores_raw.mean(), scores_pca.mean(), scores_pca2.mean()]
stds = [scores_raw.std(), scores_pca.std(), scores_pca2.std()]
colors = [ML_BLUE, ML_PURPLE, ML_ORANGE]

bars = ax.bar(labels, means, yerr=stds, color=colors, alpha=0.7,
              capsize=8, edgecolor='white', linewidth=1.5)

for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{m:.3f}', ha='center', fontsize=12, fontweight='bold')

ax.set_ylabel('5-Fold CV Accuracy')
ax.set_title('Logistic Regression Accuracy: Raw Features vs. PCA Preprocessing')
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f'Raw (50 features): {scores_raw.mean():.3f} +/- {scores_raw.std():.3f}')
print(f'PCA 95% ({pca_pre.n_components_} PCs): {scores_pca.mean():.3f} +/- {scores_pca.std():.3f}')
print(f'PCA 2 PCs: {scores_pca2.mean():.3f} +/- {scores_pca2.std():.3f}')

## Summary

**Key Takeaways:**
- **PCA finds directions of maximum variance** by computing eigenvectors of the covariance matrix -- the first few PCs often capture most of the signal
- **The scree plot guides component selection**: keep enough PCs to explain 90-95% of variance, or use cross-validation on a downstream task
- **PCA is linear only**: it fails on curved manifolds like the Swiss roll -- use t-SNE or UMAP for non-linear structure
- **PCA as preprocessing** removes noise and multicollinearity, often matching or improving classification accuracy with far fewer features